# Anomaly Detection Training
Train an LSTM autoencoder on synthetic vitals windows, compute threshold, update vocab, convert to TFLite.

In [1]:
import numpy as np
from pathlib import Path
import json
from tensorflow import keras

ROOT = Path('..')
DATA_DIR = ROOT / 'data' / 'processed' / 'anomaly'
X = np.load(DATA_DIR / 'X_train.npy')
# shape: (n_windows, 10, 4)
n_timesteps = X.shape[1]
n_features = X.shape[2]

ModuleNotFoundError: No module named 'numpy'

In [ ]:
encoder_inputs = keras.layers.Input(shape=(n_timesteps, n_features))
x = keras.layers.LSTM(64, return_sequences=True)(encoder_inputs)
x = keras.layers.LSTM(32, return_sequences=True)(x)
encoded = keras.layers.LSTM(16, return_sequences=False)(x)
# decoder that returns sequences
x = keras.layers.RepeatVector(n_timesteps)(encoded)
x = keras.layers.LSTM(16, return_sequences=True)(x)
x = keras.layers.LSTM(32, return_sequences=True)(x)
decoded = keras.layers.LSTM(64, return_sequences=True)(x)
decoded = keras.layers.TimeDistributed(keras.layers.Dense(n_features))(decoded)
model = keras.Model(encoder_inputs, decoded)
model.compile(optimizer='adam', loss='mse')
model.summary()
model.fit(X, X, epochs=100, batch_size=64, validation_split=0.1)

In [ ]:
# Compute reconstruction errors and threshold
recon = model.predict(X)
errors = np.mean(np.square((X - recon)), axis=(1,2))
threshold = float(np.mean(errors) + 2 * np.std(errors))
print('Computed threshold', threshold)
# Update vocab file
vocab_path = ROOT / 'assets' / 'symptom_vocab.json'
if vocab_path.exists():
    d = json.loads(vocab_path.read_text())
else:
    d = {}
d['anomaly_threshold'] = threshold
vocab_path.parent.mkdir(parents=True, exist_ok=True)
vocab_path.write_text(json.dumps(d, indent=2))
print('Updated vocab anomaly_threshold')

In [ ]:
# Convert to TFLite
import tensorflow as tf
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
(Path('..') / 'models').mkdir(parents=True, exist_ok=True)
with open(Path('..') / 'models' / 'anomaly_detection_v1.tflite', 'wb') as f:
    f.write(tflite_model)
print('Saved models/anomaly_detection_v1.tflite')